In [ ]:
import pandas as pd
import numpy as np

# =========================
# 1. Load Dataset
# =========================

df = pd.read_excel("pkmndataset.xlsx")

# Committee override: Toxtricity was incorrectly marked as not final evolution
df.loc[df["Name"].eq("Toxtricity"), "Final Evolution"] = 1


# =========================
# 2. Filter Study Population
# =========================

study = df[
    (df["Final Evolution"] == 1) &
    (df["Legendary"] == 0) &
    (df["Mega Evolution"] == 0)
].copy()


# =========================
# 3. Base Hotness Score
# =========================

hotness = np.full(len(study), 5.0)

for t in ["Type 1", "Type 2"]:
    hotness += study[t].isin(["Fire", "Fairy"]).astype(int) * 2
    hotness += study[t].isin(["Psychic", "Dark", "Ghost"]).astype(int)
    hotness -= study[t].isin(["Bug"]).astype(int) * 2
    hotness -= study[t].isin(["Rock", "Poison"]).astype(int)

hotness += (study["Att"] > 100).astype(int)
hotness += (study["Spe"] > 100).astype(int)
hotness += (study["BST"] > 550).astype(int)

hotness = np.clip(hotness, 0, 10)


# =========================
# 4. Base Personality Score
# =========================

personality = np.full(len(study), 5.0)

for t in ["Type 1", "Type 2"]:
    personality += study[t].isin(["Psychic"]).astype(int) * 2
    personality += study[t].isin(["Ghost", "Dragon", "Dark", "Fairy"]).astype(int)
    personality -= study[t].isin(["Normal", "Flying"]).astype(int)

against_cols = [c for c in study.columns if str(c).startswith("Against ")]

if against_cols:
    weaknesses = (study[against_cols] > 1).sum(axis=1)
    personality += (weaknesses >= 6).astype(int)
    personality -= (weaknesses <= 2).astype(int)

# Catch rate dating psychology
personality -= (study["Catch Rate"] < 45).astype(int)   # hard to get
personality -= (study["Catch Rate"] > 200).astype(int)  # needy

personality = np.clip(personality, 0, 10)


# =========================
# 5. Base Utility Score
# =========================

utility = np.full(len(study), 5.0)

utility += (study["BST"] >= 500).astype(int)
utility += (study["BST"] >= 550).astype(int)
utility += (study["BST"] >= 600).astype(int)

utility += (study["Height"] > 1.8).astype(int)
utility += (study["Height"] > 2.5).astype(int)

utility += (study["Weight"] > 100).astype(int)
utility += (study["Weight"] > 250).astype(int)

utility += (
    study["Type 1"].eq("Flying") |
    study["Type 2"].eq("Flying")
).astype(int)

utility = np.clip(utility, 0, 10)


# =========================
# 6. Synergy Modifiers
# =========================

hotness_tax = (hotness >= 8).astype(int)
utility -= hotness_tax

freak_bonus = (personality >= 7).astype(int)
utility += freak_bonus

main_character_penalty = ((hotness + personality) >= 15).astype(int)
utility -= main_character_penalty * 2

high_maintenance = (
    (study["BST"] >= 600) |
    (study["Catch Rate"] < 45)
).astype(int)
utility -= high_maintenance

edge_type = (
    study["Type 1"].isin(["Ghost", "Dark", "Dragon"]) |
    study["Type 2"].isin(["Ghost", "Dark", "Dragon"])
)

edgelord_tax = ((personality >= 7) & edge_type).astype(int)
utility -= edgelord_tax

costco_test = (utility > 7).astype(int)
hotness += costco_test

scrungler_factor = (
    (personality.isin([5, 6])) &
    (utility == 6)
).astype(int)

hotness += scrungler_factor * 2

hotness = np.clip(hotness, 0, 10)
utility = np.clip(utility, 0, 10)


# =========================
# 7. Final Dataset
# =========================

overall = hotness + personality + utility

scored = study[["Number", "Name", "Type 1", "Type 2"]].copy()

scored["Hotness"] = hotness
scored["Personality"] = personality
scored["Utility"] = utility
scored["Overall"] = overall

scored["Hotness_Tax"] = hotness_tax
scored["Freak_Bonus"] = freak_bonus
scored["Main_Character_Penalty"] = main_character_penalty
scored["High_Maintenance"] = high_maintenance
scored["Edgelord_Tax"] = edgelord_tax
scored["Costco_Test"] = costco_test
scored["Scrungler_Factor"] = scrungler_factor


# =========================
# 8. Manual Committee Override
# =========================

tox_mask = scored["Name"].eq("Toxtricity")

scored.loc[tox_mask, "Personality"] = 5
scored.loc[tox_mask, "Utility"] = 6
scored.loc[tox_mask, "Scrungler_Factor"] = 1
scored.loc[tox_mask, "Hotness"] = 6
scored.loc[tox_mask, "Overall"] = 17


# =========================
# 9. Leaderboards
# =========================

top_10_overall = scored.sort_values("Overall", ascending=False).head(10)
top_10_hotness = scored.sort_values("Hotness", ascending=False).head(10)
top_10_personality = scored.sort_values("Personality", ascending=False).head(10)
top_10_utility = scored.sort_values("Utility", ascending=False).head(10)

best_by_primary_type = (
    scored.sort_values("Overall", ascending=False)
    .groupby("Type 1", as_index=False)
    .first()
    .sort_values("Overall", ascending=False)
)


# =========================
# 10. Export Workbook
# =========================

with pd.ExcelWriter("Pokemon_Dating_Report_2026_Final.xlsx", engine="openpyxl") as writer:
    scored.to_excel(writer, sheet_name="All_Scored_Pokemon", index=False)
    top_10_overall.to_excel(writer, sheet_name="Top10_Overall", index=False)
    top_10_hotness.to_excel(writer, sheet_name="Top10_Hotness", index=False)
    top_10_personality.to_excel(writer, sheet_name="Top10_Personality", index=False)
    top_10_utility.to_excel(writer, sheet_name="Top10_Utility", index=False)
    best_by_primary_type.to_excel(writer, sheet_name="Best_By_Primary_Type", index=False)

print("Done! Workbook created: Pokemon_Dating_Report_2026_Final.xlsx")

In [1]:
# =========================
# 11. Visualizer
# =========================
import pandas as pd
import plotly.graph_objects as go

# Load workbook
input_path = "Pokemon_Dating_Report_2026_With_Toxtricity_Added.xlsx"
df = pd.read_excel(input_path, sheet_name="All_Scored_Pokemon")

factor_cols = [
    "Hotness_Tax",
    "Freak_Bonus",
    "Main_Character_Penalty",
    "High_Maintenance",
    "Edgelord_Tax",
    "Costco_Test",
    "Scrungler_Factor",
]

for col in factor_cols:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

for col in ["Hotness", "Personality", "Utility", "Overall"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["Type 1"] = df["Type 1"].fillna("Unknown")
df["Type 2"] = df["Type 2"].fillna("")

df["Active_Factors"] = df[factor_cols].apply(
    lambda row: ", ".join(
        [col.replace("_", " ") for col in factor_cols if row[col] == 1]
    ) or "None",
    axis=1
)

# Aggregate Pokémon that share the exact same H/P/U score
grouped = (
    df.groupby(["Hotness", "Personality", "Utility"], as_index=False)
    .agg(
        Pokemon_Count=("Name", "count"),
        Pokemon_List=("Name", lambda x: "<br>".join(sorted(x.astype(str)))),
        Average_Overall=("Overall", "mean"),
        Type_List=("Type 1", lambda x: ", ".join(sorted(set(x.astype(str))))),
        Factor_List=("Active_Factors", lambda x: "<br>".join(sorted(set(x.astype(str)))))
    )
)

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=grouped["Hotness"],
        y=grouped["Personality"],
        z=grouped["Utility"],
        mode="markers",
        marker=dict(
            size=grouped["Pokemon_Count"] * 2 + 5,
            color=grouped["Average_Overall"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Avg Overall"),
            opacity=0.85,
        ),
        customdata=grouped[
            ["Pokemon_Count", "Pokemon_List", "Average_Overall", "Type_List", "Factor_List"]
        ],
        hovertemplate=(
            "<b>Score cluster</b><br>"
            "Hotness: %{x}<br>"
            "Personality: %{y}<br>"
            "Utility: %{z}<br>"
            "Pokémon count: %{customdata[0]}<br>"
            "Average overall: %{customdata[2]:.1f}<br><br>"
            "<b>Pokémon here:</b><br>%{customdata[1]}<br><br>"
            "<b>Primary types:</b><br>%{customdata[3]}<br><br>"
            "<b>Factors present:</b><br>%{customdata[4]}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Pokémon Dating Report 2026 — 3D Score Clusters",
    scene=dict(
        xaxis_title="Hotness",
        yaxis_title="Personality",
        zaxis_title="Utility",
        xaxis=dict(range=[0, 10]),
        yaxis=dict(range=[0, 10]),
        zaxis=dict(range=[0, 10]),
    ),
    margin=dict(l=0, r=0, b=0, t=60),
)

fig.show()

# Optional export
fig.write_html("Pokemon_Dating_Report_2026_3D_Clustered.html", include_plotlyjs="cdn")


FileNotFoundError: [Errno 2] No such file or directory: 'Pokemon_Dating_Report_2026_With_Toxtricity_Added.xlsx'